# Flujo de información para generar capa comunal
#### Modelo de estimación de demanda eléctrica MERLIN_EDM
--- 
En este Notebook se empleará el modelo ``merlin_edm`` entrenado pra obtener la construcción de capas geoespaciales de demanda eléctrica en escala comunal y desagregada por sector de interés. 

Los archivos de entrada para esto serán: 

- ``../data/interim/shares_comunales.parquet``: Los shares comunales provenientes de unir los datos de facturación de clientes regulados, con la venta de energía a clientes libres. Este archivo se encuentra en formato ancho, es decir, tiene: ``"año_ms" | "año" | "region"| "comuna" | f"consumo_{sector}_MWh" for sector in SECTORES | "consumo_total_mes_MWh"``
- ``../data/rec_2024_2025/temperatura_comunal_2024_2025.parquet``: Es la temperatura de los años 2024 y 2025 en todo Chile, en escala comunal y resolución horaria. 
- ``../data/raw/reg_alias.json``: Son los alias de formato ISO de la región con respecto a su nombre disponible en las bases geoespaciales. Sirve para cruzar la información del balance regional de energía con el de las temperaturas. 

El procesamiento de estos archivos de entrada llevarán a lo siguiente: 

- Generación de rezagos temporales de temperatura (temperatura presente y 7 lags hacia atrás).
- Cálculo de series trigonométricas de hora/semana/año. 
- Condicionales de día hábil/fin de semana/feriado en escala regional. 
- Intensidades energéticas en escala mensual ``total_comunal/total_nacional`` y por sector ``total_sector_comuna/total_comunal``.

Se debería generar una matriz de inputs que contenga los datos de todo el país para poder tomarlo como inferencia del modelo. El orden de los inputs importa para la red neuronal. Este se guarda en un archivo de texto llamado ``../data/rec_2024_2025/columns.txt`` y para cargarlo a la sesión el código es: 

```python
columns = []  # Lista vacía para que se guarden los nombres de las columnas
with open("../data/rec_2024_2025/columns.txt", "r") as f:
    for line in f: 
        columns.append(line.strip()) 

```

La matriz de inputs se usa para el modelo de red neuronal entrenado. Los outputs que se deberían obtener son: 

- ``../data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal.gpkg``: Geocapa con los totales anuales (año 2024 y 2025) en cada comuna, total y por sector (RCPIT)
- ``../data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal_ts.parquet``: Serie de tiempo con la demanda de electricidad (año 2024 y 2025) en cada comuna, total y por sector (RCPIT)

Hay que hacer un pequeño procesamiento previo para hacer la inferencia, y que corresponde a extrapolar los consumos mensuales comunales a otras comunas faltantes. Luego de la generación de la base ``shares_comunales.parquet``, quedó información de desagregación de consumos por sector en 212 de 345 comunas. Para hacer la extrapolación de esos consumos, se considerará el tamaño y cercanía de las comunas. Entonces, dada una comuna $j$ sin información de consumos, se buscará la vecina $i$ más cercana que presente área similar y se copiarán sus consumos. O bien, se calculará el promedio de consumos de todos sus vecinos.

In [1]:
# ==========================================
# CELDA 1: CONFIGURACIÓN E IMPORTACIONES
# ==========================================
import os
import sys
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
import numpy as np
import matplotlib.pyplot as plt
import holidays
import calendar
import geopandas as gpd
import warnings
import unicodedata

# 1. Configuración de Rutas Globales
BASE_DIR = os.path.abspath("..")
MODEL_PATH = os.path.join(BASE_DIR, "models", "ds_comunal", "best_merlin_mlp_global.keras")
LAYER_PATH = os.path.join(BASE_DIR, "data", "raw", "capa_comunal.gpkg")
TEMP_SCALER_PATH = os.path.join(BASE_DIR, "models", "scaler_temp_global.pkl")
HIST_SHARES_PATH = os.path.join(BASE_DIR, "data", "interim", "shares_comunales.parquet")
TEMP_REG_PATH = os.path.join(BASE_DIR, "data", "rec_2024_2025", "temperatura_comunal_2024_2025.parquet")
COLUMNS_PATH = os.path.join(BASE_DIR, "data", "rec_2024_2025", "columns.txt")
OUT_DIR = os.path.join(BASE_DIR, "data", "rec_2024_2025", "results", "capas_comunales")

os.makedirs(OUT_DIR, exist_ok=True)

# 2. Parámetros del Modelo
IS_COMUNA = 1  # Trabajamos con comunas ahora
SECTORES = ['I', 'R', 'C', 'P', 'T']
A_PARAM = np.exp(-1.1315)  
B_PARAM = 0.8988
AÑOS_TARGET = [2024, 2025]

# 3. Cargar columnas
columns = []  # Lista vacía para que se guarden los nombres de las columnas
with open("../data/rec_2024_2025/columns.txt", "r") as f:
    for line in f: 
        columns.append(line.strip()) 

# 4. Cargar modelos de red neuronal y scaler de temperatura
model = load_model(MODEL_PATH)
temp_scaler = joblib.load(TEMP_SCALER_PATH)

# 5. Cargar los shares regionales
df_hist_shares = pd.read_parquet(HIST_SHARES_PATH)

# 6. Cargar los shares de temperatura
df_temp_global = pd.read_parquet(TEMP_REG_PATH)
comunas = df_temp_global["comuna"].unique().tolist()

I0000 00:00:1784823152.513752 3560656 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784823152.543028 3560656 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1784823157.990097 3560656 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1784823161.612145 3560656 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries men

In [2]:
# ==========================================
# CELDA 2: CARGA GEOGRÁFICA Y EVALUACIÓN DE COBERTURA
# ==========================================
print("1. Cargando la capa geográfica y estandarizando coordenadas...")
gdf_mapa = gpd.read_file(LAYER_PATH)

# Asegurarse de usar una proyección en metros (ej. EPSG:32719 UTM 19S para Chile)
# Esto es vital para calcular áreas exactas y distancias métricas
if gdf_mapa.crs != "EPSG:32719":
    gdf_mapa = gdf_mapa.to_crs("EPSG:32719")

# Calcular los centroides geométricos y el área física (en km2)
gdf_mapa['centroide'] = gdf_mapa.geometry.centroid
gdf_mapa['area_km2'] = gdf_mapa.geometry.area / 10**6

def limpiar_texto(texto):
    if pd.isna(texto):
        return texto
    # Normaliza a formato NFKD, codifica a ASCII ignorando caracteres especiales (tildes) y decodifica
    texto_limpio = unicodedata.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('utf-8')
    return texto_limpio.strip().upper()

# 2. Homogeneizar textos para el cruce (mayúsculas y sin espacios extra)
df_hist_shares['comuna'] = df_hist_shares['comuna'].apply(limpiar_texto)
gdf_mapa['comuna'] = gdf_mapa['comuna'].apply(limpiar_texto)
comunas_gdf = gdf_mapa['comuna'].unique().tolist()
comunas_totales_lista = [str(c).strip().upper() for c in comunas_gdf]

# 3. Separar los universos: las 212 con datos y las faltantes
comunas_con_datos = df_hist_shares['comuna'].unique().tolist()
comunas_faltantes = [c for c in comunas_totales_lista if c not in comunas_con_datos]

print(f"Total comunas mapeadas: {len(comunas_totales_lista)}")
print(f"Comunas con datos históricos: {len(comunas_con_datos)}")
print(f"Comunas a extrapolar: {len(comunas_faltantes)}")

1. Cargando la capa geográfica y estandarizando coordenadas...
Total comunas mapeadas: 345
Comunas con datos históricos: 212
Comunas a extrapolar: 133


In [4]:
# ==========================================
# CELDA 3: MOTOR DE EXTRAPOLACIÓN ESPACIAL (IDW + ÁREA Y PRÉSTAMO DE PERFIL)
# ==========================================

warnings.filterwarnings('ignore')

print("2. Iniciando extrapolación espacial (Tamaño, Cercanía y Perfil Regional/Donante)...")

# =================================================================
# NUEVO BLOQUE: INTERPOLACIÓN DE MESES FALTANTES CON AÑO DONANTE
# =================================================================
print(" -> Pre-procesando: Calculando perfiles estacionales regionales y buscando años donantes...")

# 1. Asegurar formato datetime para operaciones temporales
df_hist_shares['año_ms'] = pd.to_datetime(df_hist_shares['año_ms'], errors='coerce')
cols_consumo = [f"consumo_{s}_MWh" for s in SECTORES]

# --- PASO A: CALCULAR EL PERFIL DE LA REGIÓN ---
df_region = df_hist_shares.groupby(['año_ms', 'region'])[cols_consumo].sum().reset_index()
df_region['año'] = df_region['año_ms'].dt.year

for col in cols_consumo:
    total_anual_reg = df_region.groupby(['region', 'año'])[col].transform('sum')
    df_region[f'peso_{col}'] = (df_region[col] / total_anual_reg.replace(0, pd.NA)).fillna(1/12)

print(" -> Pre-procesando: Imputando meses faltantes por préstamo de año completo (ej. 2020 para completar 2021)...")

dfs_completos = []
for comuna, df_grupo in df_hist_shares.groupby('comuna'):
    # Ordenar por fecha y fijar como índice para usar resample
    df_grupo = df_grupo.set_index('año_ms').sort_index()
    df_grupo = df_grupo[~df_grupo.index.duplicated(keep='first')]
    
    # Remuestrear a nivel mensual (MS) para crear NaNs en los meses faltantes
    df_resampled = df_grupo.resample('MS').asfreq().reset_index()
    
    # Restaurar metadatos básicos
    df_resampled['comuna'] = comuna
    region_actual = df_grupo['region'].dropna().iloc[0] if not df_grupo['region'].dropna().empty else "SIN_REGION"
    df_resampled['region'] = region_actual
    df_resampled['año'] = df_resampled['año_ms'].dt.year
    
    # --- PASO B: CRUZAR CON EL PERFIL REGIONAL ---
    df_resampled = pd.merge(
        df_resampled, 
        df_region[['año_ms', 'region'] + [f'peso_{c}' for c in cols_consumo]], 
        on=['año_ms', 'region'], 
        how='left'
    )
    
    # --- PASO C: IMPUTAR HUECOS USANDO AÑO DONANTE COMPLETO O PESOS REGIONALES ---
    for col in cols_consumo:
        if col in df_resampled.columns:
            
            # Identificar qué años tienen los 12 meses completos y sin nulos (Ej: 2020)
            anios_completos = []
            for a in df_resampled['año'].dropna().unique():
                df_a = df_resampled[df_resampled['año'] == a]
                if len(df_a) == 12 and df_a[col].notna().all():
                    anios_completos.append(a)
            
            # Iterar año por año para revisar si hay huecos (como en 2021)
            for año in df_resampled['año'].dropna().unique():
                
                # Usamos máscaras nativas de Pandas (sin .values) para no perder los índices
                df_año = df_resampled[df_resampled['año'] == año]
                
                mask_validos = df_año[col].notna()
                mask_nans = df_año[col].isna()
                
                if mask_nans.any():
                    # Índices exactos donde están los NaNs para este año particular
                    indices_nans = df_año[mask_nans].index
                    
                    # CASO 1: Tenemos meses parciales (ej. Ene-Jun 2021) y un año completo donante (ej. 2020)
                    if mask_validos.any() and len(anios_completos) > 0:
                        año_donante = anios_completos[0]
                        df_donante = df_resampled[df_resampled['año'] == año_donante]
                        
                        # Meses válidos actuales (ej. 1 al 6)
                        meses_validos = df_año.loc[mask_validos, 'año_ms'].dt.month.values
                        
                        # Suma en esos mismos meses en el año donante y en el año actual
                        suma_donante_parcial = df_donante[df_donante['año_ms'].dt.month.isin(meses_validos)][col].sum()
                        consumo_conocido = df_año.loc[mask_validos, col].sum()
                        
                        if pd.notna(suma_donante_parcial) and suma_donante_parcial > 0:
                            factor_escala = consumo_conocido / suma_donante_parcial
                            
                            # Rellenar los meses faltantes usando el perfil del donante escalado
                            for idx_row in indices_nans:
                                mes_faltante = df_resampled.loc[idx_row, 'año_ms'].month
                                val_donante = df_donante[df_donante['año_ms'].dt.month == mes_faltante][col].values
                                if len(val_donante) > 0 and pd.notna(val_donante[0]):
                                    df_resampled.loc[idx_row, col] = val_donante[0] * factor_escala
                    
                    # CASO 2: El año está 100% vacío, replicamos el año donante completo sin escalar
                    elif not mask_validos.any() and len(anios_completos) > 0:
                        año_donante = anios_completos[0]
                        df_donante = df_resampled[df_resampled['año'] == año_donante]
                        
                        for idx_row in indices_nans:
                            mes_faltante = df_resampled.loc[idx_row, 'año_ms'].month
                            val_donante = df_donante[df_donante['año_ms'].dt.month == mes_faltante][col].values
                            if len(val_donante) > 0 and pd.notna(val_donante[0]):
                                df_resampled.loc[idx_row, col] = val_donante[0]
                    
                    # CASO 3: Fallback por pesos regionales si no hay año donante
                    else:
                        pesos_completos = df_año[f'peso_{col}'].notna().all()
                        if mask_validos.any() and pesos_completos:
                            consumo_conocido = df_año.loc[mask_validos, col].sum()
                            peso_conocido = df_año.loc[mask_validos, f'peso_{col}'].sum()
                            if peso_conocido > 0:
                                total_anual_est = consumo_conocido / peso_conocido
                                # Calcula los valores imputados alineados con los NaNs de ese año
                                val_imp = total_anual_est * df_año.loc[mask_nans, f'peso_{col}'].values
                                # Se asigna directamente a los índices correspondientes en el dataframe principal
                                df_resampled.loc[indices_nans, col] = val_imp
            
            # Salvavidas final por si queda algún NaN aislado
            df_resampled[col] = df_resampled[col].interpolate(method='linear').bfill().ffill()
            
    # Limpiar columnas temporales de cálculo
    cols_to_drop = [f'peso_{c}' for c in cols_consumo]
    df_resampled = df_resampled.drop(columns=[c for c in cols_to_drop if c in df_resampled.columns])
    
    # Recalcular el total mensual interpolado por balance de energía
    if 'consumo_total_mes_MWh' in df_resampled.columns:
        df_resampled['consumo_total_mes_MWh'] = df_resampled[cols_consumo].sum(axis=1)
        
    dfs_completos.append(df_resampled)

# Sobrescribir el dataframe histórico con la versión continua
df_hist_shares = pd.concat(dfs_completos, ignore_index=True)
print(" -> ¡Interpolación histórica con año donante completada con éxito!")
# =================================================================

# =================================================================
# EXTRAPOLACIÓN DE COMUNAS COMPLETAMENTE FALTANTES (IDW + ÁREA)
# =================================================================
gdf_con_datos = gdf_mapa[gdf_mapa['comuna'].isin(comunas_con_datos)].copy()
gdf_faltantes = gdf_mapa[gdf_mapa['comuna'].isin(comunas_faltantes)].copy()

dfs_extrapolados = []

for idx, row in gdf_faltantes.iterrows():
    com_faltante = row['comuna']
    region_faltante = row['region'] if 'region' in row else "SIN_REGION" 
    area_faltante = row['area_km2']
    centroide_faltante = row['centroide']
    
    gdf_con_datos['distancia_m'] = gdf_con_datos['centroide'].distance(centroide_faltante)
    
    vecinos = gdf_con_datos.nsmallest(3, 'distancia_m')
    df_vecinos = df_hist_shares[df_hist_shares['comuna'].isin(vecinos['comuna'])].copy()
    df_vecinos = df_vecinos.merge(vecinos[['comuna', 'area_km2', 'distancia_m']], on='comuna', how='left')
    
    df_vecinos['peso'] = 1.0 / (df_vecinos['distancia_m'] + 1.0) 
    
    def estimar_mes(grupo):
        pesos_norm = grupo['peso'] / grupo['peso'].sum()
        fila = {}
        consumo_mensual_total = 0.0
        
        for col in cols_consumo:
            densidad_vecino = grupo[col] / grupo['area_km2']
            densidad_promedio = (densidad_vecino * pesos_norm).sum()
            consumo_est = densidad_promedio * area_faltante
            
            fila[col] = consumo_est
            consumo_mensual_total += consumo_est
            
        fila['consumo_total_mes_MWh'] = consumo_mensual_total
        return pd.Series(fila)

    df_estimado = df_vecinos.groupby('año_ms').apply(estimar_mes).reset_index()
    
    df_estimado['año_ms'] = pd.to_datetime(df_estimado['año_ms'], errors='coerce')
    df_estimado['año'] = df_estimado['año_ms'].dt.year.astype('Int64') 
    df_estimado['mes'] = df_estimado['año_ms'].dt.month.astype('Int64')
    
    df_estimado['comuna'] = com_faltante
    df_estimado['region'] = region_faltante
    
    dfs_extrapolados.append(df_estimado)

if len(dfs_extrapolados) > 0:
    df_hist_extrapolado = pd.concat(dfs_extrapolados, ignore_index=True)
    df_hist_shares["mes"] = df_hist_shares["año_ms"].dt.month
    df_shares_completo = pd.concat([df_hist_shares, df_hist_extrapolado], ignore_index=True)
else:
    df_hist_shares["mes"] = df_hist_shares["año_ms"].dt.month
    df_shares_completo = df_hist_shares.copy()

# --- AISLAR Y RECONSTRUIR SOLO LOS MESES FALTANTES DE 2021 CON IDW ESPACIAL ---
print(" -> Aplicando parche espacial IDW para los meses faltantes de 2021 en la comuna afectada...")

comuna_afectada = "MARIA ELENA" 

mask_comuna_2021_hosp = (df_hist_shares['comuna'] == comuna_afectada) & (df_hist_shares['año_ms'].dt.year == 2021)
df_comuna_2021 = df_hist_shares[mask_comuna_2021_hosp]

if len(df_comuna_2021) < 12:
    row_comuna = gdf_mapa[gdf_mapa['comuna'] == comuna_afectada].iloc[0]
    centroide_obj = row_comuna['centroide']
    area_obj = row_comuna['area_km2']
    region_obj = row_comuna['region']
    
    comunas_sanas_2021 = df_hist_shares[df_hist_shares['año_ms'] >= '2021-07-01']['comuna'].unique()
    gdf_sanas = gdf_mapa[gdf_mapa['comuna'].isin(comunas_sanas_2021)].copy()
    gdf_sanas['distancia_m'] = gdf_sanas['centroide'].distance(centroide_obj)
    
    vecinos = gdf_sanas.nsmallest(3, 'distancia_m')
    df_vecinos = df_hist_shares[df_hist_shares['comuna'].isin(vecinos['comuna'])].copy()
    df_vecinos = df_vecinos.merge(vecinos[['comuna', 'area_km2', 'distancia_m']], on='comuna', how='left')
    df_vecinos['peso_idw'] = 1.0 / (df_vecinos['distancia_m'] + 1.0)
    
    meses_faltantes_dates = pd.date_range(start='2021-07-01', end='2021-12-01', freq='MS')
    
    for fecha in meses_faltantes_dates:
        df_v_mes = df_vecinos[df_vecinos['año_ms'] == fecha]
        if not df_v_mes.empty:
            pesos_norm = df_v_mes['peso_idw'] / df_v_mes['peso_idw'].sum()
            fila_estimada = {'año_ms': fecha, 'año': 2021, 'mes': fecha.month, 'comuna': comuna_afectada, 'region': region_obj}
            
            consumo_mes_total = 0.0
            for col in cols_consumo:
                densidad_vecino = df_v_mes[col] / df_v_mes['area_km2']
                densidad_promedio = (densidad_vecino * pesos_norm).sum()
                consumo_est = densidad_promedio * area_obj
                fila_estimada[col] = consumo_est
                consumo_mes_total += consumo_est
                
            fila_estimada['consumo_total_mes_MWh'] = consumo_mes_total
            
            idx_existente = df_hist_shares[(df_hist_shares['comuna'] == comuna_afectada) & (df_hist_shares['año_ms'] == fecha)].index
            if len(idx_existente) > 0:
                for k, v in fila_estimada.items():
                    df_hist_shares.loc[idx_existente, k] = v
            else:
                df_hist_shares = pd.concat([df_hist_shares, pd.DataFrame([fila_estimada])], ignore_index=True)

# Actualizar el completo despues del bucle de IDW (mucho mas rápido y eficiente)
df_shares_completo = pd.concat([df_hist_shares, df_hist_extrapolado], ignore_index=True) if len(dfs_extrapolados) > 0 else df_hist_shares.copy()

# ==========================================================
# PARCHE MANUAL EXCLUSIVO PARA: MARIA ELENA (2021 - JULIO A DICIEMBRE)
# ==========================================================

print(" -> Aplicando parche manual de control para María Elena (2021 - 2do Semestre)...")

cols_sectores_nulos = [f"consumo_{s}_MWh" for s in ["R", "C", "P", "T"]]
col_industrial = "consumo_I_MWh"

for col in cols_sectores_nulos + [col_industrial]:
    if col not in df_shares_completo.columns:
        df_shares_completo[col] = np.nan

mask_maria_elena = df_shares_completo["comuna"].astype(str).str.upper() == "MARIA ELENA"
mask_2021_segundo_semestre = (df_shares_completo["año"] == 2021) & (df_shares_completo["año_ms"].dt.month >= 7)

mask_objetivo_2021_2do = mask_maria_elena & mask_2021_segundo_semestre

df_shares_completo.loc[mask_objetivo_2021_2do, cols_sectores_nulos] = np.nan

for mes in range(7, 13):
    mes_origen = mes - 6
    mask_origen = mask_maria_elena & (df_shares_completo["año"] == 2021) & (df_shares_completo["año_ms"].dt.month == mes_origen)
    mask_destino = mask_maria_elena & (df_shares_completo["año"] == 2021) & (df_shares_completo["año_ms"].dt.month == mes)
    
    if mask_origen.any() and mask_destino.any():
        val_industrial = df_shares_completo.loc[mask_origen, col_industrial].values[0]
        df_shares_completo.loc[mask_destino, col_industrial] = val_industrial

cols_todos_sectores = [f"consumo_{s}_MWh" for s in ["R", "C", "P", "I", "T"]]
if "consumo_total_mes_MWh" in df_shares_completo.columns:
    df_shares_completo.loc[mask_objetivo_2021_2do, "consumo_total_mes_MWh"] = (
        df_shares_completo.loc[mask_objetivo_2021_2do, cols_todos_sectores].sum(axis=1, min_count=1)
    )

print(" -> ¡Parche manual aplicado con éxito para María Elena (2021 - Julio a Diciembre)! Nulos, herencia industrial y totales actualizados.")

# ==========================================================
# REASIGNACIÓN REGIONAL MANUAL: COMUNAS DE ÑUBLE
# ==========================================================

print(" -> Corrigiendo asignación regional para comunas de la Región de Ñuble...")

comunas_nuble_target = ["YUNGAY", "SAN CARLOS", "CHILLAN", "CHILLÁN", "NIQUEN", "ÑIQUÉN"]
nombre_region_nuble = "ÑUBLE" 

mask_comunas_nuble = (
    df_shares_completo["comuna"]
    .astype(str)
    .str.upper()
    .str.strip()
    .isin(comunas_nuble_target)
)

df_shares_completo.loc[mask_comunas_nuble, "region"] = nombre_region_nuble

print(f" -> ¡Corrección aplicada! Se actualizaron {mask_comunas_nuble.sum()} filas correspondientes a Yungay, San Carlos, Chillán y Ñiquén.")

print(f"Extrapolación finalizada con éxito.")
print(f"Dataset consolidado final cuenta con información de {len(df_shares_completo['comuna'].unique())} comunas.")

2. Iniciando extrapolación espacial (Tamaño, Cercanía y Perfil Regional/Donante)...
 -> Pre-procesando: Calculando perfiles estacionales regionales y buscando años donantes...
 -> Pre-procesando: Imputando meses faltantes por préstamo de año completo (ej. 2020 para completar 2021)...
 -> ¡Interpolación histórica con año donante completada con éxito!
 -> Aplicando parche espacial IDW para los meses faltantes de 2021 en la comuna afectada...
 -> Aplicando parche manual de control para María Elena (2021 - 2do Semestre)...
 -> ¡Parche manual aplicado con éxito para María Elena (2021 - Julio a Diciembre)! Nulos, herencia industrial y totales actualizados.
 -> Corrigiendo asignación regional para comunas de la Región de Ñuble...
 -> ¡Corrección aplicada! Se actualizaron 192 filas correspondientes a Yungay, San Carlos, Chillán y Ñiquén.
Extrapolación finalizada con éxito.
Dataset consolidado final cuenta con información de 345 comunas.


### Extrapolación de los shares en el tiempo
TBD

In [5]:
# ==========================================
# CELDA 4: EXTRAPOLACIÓN DE SHARES Y RATIOS (CORREGIDA)
# ==========================================

print("Iniciando extrapolación de consumos y cálculo de shares por mes...")

sectores = ['Industrial', 'Residencial', 'Comercial', 'Público', 'Transporte']
cols_sectores = [f"consumo_{sector}_MWh" for sector in SECTORES]

proyecciones = []
comunas = df_shares_completo["comuna"].unique().tolist()

for comuna in comunas:
    # 1. Aislamos la historia exclusiva de esta comuna y la ordenamos
    df_comuna = df_shares_completo[df_shares_completo['comuna'] == comuna].sort_values(['año', 'mes']).copy()
    
    # Guardamos todo el histórico original de la comuna en la lista
    proyecciones.append(df_comuna)
    
    # 2. Iterar por cada mes único presente en la historia de la comuna
    for mes in df_comuna["mes"].unique().tolist():
        # ¡CORRECCIÓN CRÍTICA!: Filtrar estrictamente el histórico para ESTE mes específico
        df_mes = df_comuna[df_comuna['mes'] == mes]
        years_hist = df_mes['año'].values
        
        # 3. Proyectar para los años objetivo (2024, 2025)
        for target_year in AÑOS_TARGET:
            # Si el año ya existe para este mes en los datos originales, no lo sobreescribimos
            if target_year in years_hist:
                continue
                
            nueva_fila = {'año': target_year, 'comuna': comuna, "mes": mes}
            
            # Preservar la región si existe en el DataFrame base
            if 'region' in df_mes.columns:
                nueva_fila['region'] = df_mes.iloc[0]['region']
                
            for sec in cols_sectores:
                valores_hist = df_mes[sec].values
                if len(years_hist) < 2:
                    # Si no hay historia suficiente para trazar una recta, copiamos el último valor de ese mes
                    nueva_fila[sec] = valores_hist[-1] if len(valores_hist) > 0 else 0.0
                else:
                    # Extrapolación lineal exclusiva para este sector, comuna y mes
                    z = np.polyfit(years_hist, valores_hist, 1)
                    p = np.poly1d(z)
                    # FILTRO FÍSICO: El consumo proyectado nunca puede ser negativo
                    nueva_fila[sec] = max(0.0, p(target_year))
            
            proyecciones.append(pd.DataFrame([nueva_fila]))

# 4. Unir todo el historial + el futuro en un solo DataFrame
df_shares_proyectados = pd.concat(proyecciones, ignore_index=True)

# 5. Calcular los totales y los Shares definitivos por mes y comuna
df_shares_proyectados['consumo_total_mes_MWh'] = df_shares_proyectados[cols_sectores].sum(axis=1)

for sec in SECTORES:
    sec_col = f"consumo_{sec}_MWh"
    nombre_columna = f'share_{sec}'
    
    # Fracción = Sector / Total (usando np.where para evitar división por cero)
    df_shares_proyectados[nombre_columna] = np.where(
        df_shares_proyectados['consumo_total_mes_MWh'] > 0,
        df_shares_proyectados[sec_col] / df_shares_proyectados['consumo_total_mes_MWh'],
        0.0
    )

# 6. Calcular el consumo total nacional agrupado por AÑO y MES
df_nacional = df_shares_proyectados.groupby(['año', 'mes'])['consumo_total_mes_MWh'].sum().reset_index()
df_nacional.rename(columns={'consumo_total_mes_MWh': 'total_consumo_nacional'}, inplace=True)

# ¡CORRECCIÓN CRÍTICA!: Unir el total nacional usando ambas llaves temporales ['año', 'mes']
df_shares_proyectados = pd.merge(df_shares_proyectados, df_nacional, on=['año', 'mes'], how='left')

# 7. Calcular la proporción mensual (total_comunal / total_nacional para cada mes)
df_shares_proyectados['region_comuna_share'] = np.where(
    df_shares_proyectados['total_consumo_nacional'] > 0,
    df_shares_proyectados['consumo_total_mes_MWh'] / df_shares_proyectados['total_consumo_nacional'],
    0.0
)

print("¡Proyección de shares y ratios mensuales completada con éxito!")

Iniciando extrapolación de consumos y cálculo de shares por mes...
¡Proyección de shares y ratios mensuales completada con éxito!


### Cálculo de lags de temperatura y series del calendario comunal
Acá se van a calcular (para cada comuna): 
- Los lags de temperatura que el modelo considera para la "inercia" de $\tau = 7$ horas.
- Series de sin y cos de las horas/semanas/año del intervalo de tiempo que se va a hacer forecast.

En cuanto a los indicadores que definen si es día festivo o no, se determinarán a nivel regional, ya que la librería ``holidays`` de Python tiene integrada sólo la subdivisión de Chile hasta ese nivel de desagregación espacial.



In [6]:
# PROBAR FUNCIONES DE time_features.py
def build_temperature_lags(
        df_temp, 
        temp_col='temperatura', 
        tau=7
):
    """
    Toma un Dataframe anual de temperatura y genera los lags, 
    usando las últimas 'tau' horas para rellenar el inicio.
    """

    df = df_temp.copy()

    for i in range(1, tau + 1): 
        # np.roll desplaza los valores. Al desplazar hacia abajo, 
        # los últimos valores pasan automáticamente al principio
        df[f'temp_t - {i}'] = np.roll(df[temp_col], i)

    return df


def build_calendar_features(df, dt_col='fecha_hora', country='CL', subdiv=None):
    """
    Construye las variables trigonométricas y categóricas del calendario
    a partir de una columna de fecha y hora.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene la serie temporal.
        dt_col (str): Nombre de la columna con las fechas (Datetime).
        country (str): Código ISO del país para buscar los feriados (Defecto: 'CL').
        subdiv (str): Código  ISO de la subdivisón (región) del país (Defecto: None)
        
    Returns:
        pd.DataFrame: DataFrame con las nuevas columnas de features temporales.
    """
    df = df.copy()
    
    # 0. Asegurar que la columna de entrada sea de tipo datetime de Pandas
    if not pd.api.types.is_datetime64_any_dtype(df[dt_col]):
        df[dt_col] = pd.to_datetime(df[dt_col])
        
    # 1. Extraer componentes base
    hour = df[dt_col].dt.hour
    day_of_week = df[dt_col].dt.dayofweek  # Lunes = 0, Domingo = 6
    day_of_year = df[dt_col].dt.dayofyear
    # Detectar años bisiestos para ajustar la longitud del ciclo anual
    days_in_year = df[dt_col].dt.is_leap_year.map({True: 366, False: 365})
    
    # 2. Transformaciones Trigonométricas (Ciclos)
    # Ciclo diario (24 horas)
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    
    # Ciclo semanal (7 días)
    df['dow_sin'] = np.sin(2 * np.pi * day_of_week / 7)
    df['dow_cos'] = np.cos(2 * np.pi * day_of_week / 7)
    
    # Ciclo anual (365/366 días) - *Sugerido para complementar estacionalidad
    df['doy_sin'] = np.sin(2 * np.pi * day_of_year / days_in_year)
    df['doy_cos'] = np.cos(2 * np.pi * day_of_year / days_in_year)
    
    # 3. Flags Categóricos (Fines de semana y Festivos)
    df['is_weekend'] = df[dt_col].dt.dayofweek.isin([5, 6]).astype(int)
    
    # Obtener festivos del país para los años presentes en el DataFrame
    years = df[dt_col].dt.year.unique()
    cl_holidays = holidays.country_holidays(country, subdiv=subdiv, years=years)
    
    # Mapear si la fecha cae en un día festivo
    df['is_holiday'] = df[dt_col].dt.date.apply(lambda d: d in cl_holidays).astype(int)
    
    # 4. Día laboral (Es True solo si NO es fin de semana y NO es festivo)
    df['is_working_day'] = ((df['is_weekend'] == 0) & (df['is_holiday'] == 0)).astype(int)
    
    return df


In [7]:
regiones_ISO = {
    "TARAPACÁ": "TA", 
    "ANTOFAGASTA": "AN", 
    "ATACAMA": "AT", 
    "COQUIMBO": "CO", 
    "VALPARAÍSO": "VS", 
    "LIBERTADOR GENERAL BERNARDO O'HIGGINS": "LI", 
    "MAULE": "ML", 
    "BIOBÍO": "BI", 
    "LA ARAUCANÍA": "AR", 
    "LOS LAGOS": "LL", 
    "AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO": "AI", 
    "MAGALLANES Y DE LA ANTÁRTICA CHILENA": "MA", 
    "METROPOLITANA DE SANTIAGO": "RM", 
    "LOS RÍOS": "LR", 
    "ARICA Y PARINACOTA": "AP", 
    "ÑUBLE": "NB"
}

gdf_clean = gpd.read_file(LAYER_PATH)

In [8]:
print("Generando lags de temperatura y features de calendario...")

# 1. Asegurar que la columna global sea datetime antes de entrar al bucle
df_temp_global['fecha_hora'] = pd.to_datetime(df_temp_global['fecha_hora'])
comunas_temp = df_temp_global["comuna"].unique().tolist()
# 2. Crear una lista vacía para ir guardando los pedazos de cada región
lista_comunas_procesadas = []

# Bucle for por comunas
for comuna in comunas_temp:
    print(f"Procesando características temporales para: {comuna}")
    region = gdf_clean[gdf_clean["comuna"] == comuna]["region"].values[0]
    print(f"Región: {region}")
    region_ISO = regiones_ISO[region]
    print(f"ISO: {region_ISO}")
    
    # 3. Filtrar los datos de la comuna y ORDENAR cronológicamente (CRÍTICO para np.roll)
    df_com = df_temp_global[df_temp_global["comuna"] == comuna].copy()
    df_com["region"] = region  # asignamos la columna región para cruzar con capa geoespacial 
    df_com = df_com.sort_values(by="fecha_hora").reset_index(drop=True)
    
    # 4. Generar los Lags de temperatura (temp_t-1 a temp_t-7)
    df_com = build_temperature_lags(df_com, temp_col='temperatura', tau=7)
    
    # 5. Generar las variables de calendario (Seno, Coseno, Feriados)
    # Nota: Si tu variable 'region' contiene el código ISO exacto de la región (ej: 'RM', 'AN'), 
    # puedes pasar subdiv=region. Si contiene el nombre completo, usa subdiv=None 
    # para usar los feriados nacionales de Chile ('CL').
    df_com = build_calendar_features(df_com, dt_col='fecha_hora', country='CL', subdiv=region_ISO)
    
    # 6. Almacenar el DataFrame ya procesado en la lista
    lista_comunas_procesadas.append(df_com)

# 7. Unir (Concatenar) todas las regiones procesadas en un solo DataFrame maestro
df_inputs = pd.concat(lista_comunas_procesadas, ignore_index=True)

# 8. Escalar las temperaturas (Si el modelo las espera escaladas)
# Asumiendo que temp_scaler ya fue cargado en la Celda 1
columnas_temp = ['temperatura'] + [f'temp_t - {i}' for i in range(1, 8)]
# Es importante que el scaler reciba las columnas en el mismo orden en que fue entrenado
df_inputs[columnas_temp] = temp_scaler.transform(df_inputs[columnas_temp])

print(f"¡Dataset de inputs temporales creado exitosamente! Shape: {df_inputs.shape}")

Generando lags de temperatura y features de calendario...
Procesando características temporales para: IQUIQUE
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: ALTO HOSPICIO
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: POZO ALMONTE
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: CAMIÑA
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: COLCHANE
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: HUARA
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: PICA
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: ANTOFAGASTA
Región: ANTOFAGASTA
ISO: AN
Procesando características temporales para: MEJILLONES
Región: ANTOFAGASTA
ISO: AN
Procesando características temporales para: SIERRA GORDA
Región: ANTOFAGASTA
ISO: AN
Procesando características temporales para: TALTAL
Región: ANTOFAGASTA
ISO: AN
Procesando características temporales para: CALAMA
Región: ANTOFAG

NOTA: Ñuble no está al 100% en los meses, por lo que hay que extrapolar

In [9]:
# ==========================================
# CELDA 4: FUSIÓN GLOBAL Y MATRIZ DE INPUTS EXACTA   
# ==========================================
print("Iniciando fusión global de shares proyectados y características temporales...")
# Agregar algunas características en df_inputs
df_inputs["año"] = df_inputs["fecha_hora"].dt.year.astype('Int64')
df_inputs["mes"] = df_inputs["fecha_hora"].dt.month.astype('Int64')
if "region_gdf" in df_inputs.columns:
    pass
else: 
    df_inputs["region_gdf"] = df_inputs["region"]
    df_inputs["comuna_gdf"] = df_inputs["comuna"]
    df_inputs['region'] = df_inputs['region'].apply(limpiar_texto)
    df_inputs['comuna'] = df_inputs['comuna'].apply(limpiar_texto)
# Eliminar la columna "año_ms"
if "año_ms" in df_shares_proyectados.columns:
    df_shares_proyectados.drop(columns=["año_ms"], inplace=True) 
df_shares_proyectados["region"] = df_shares_proyectados["region"].apply(limpiar_texto)

# 1. Realizar el merge masivo usando 'año', 'mes', 'region' y 'comuna' como llaves
df_master = pd.merge(df_inputs, df_shares_proyectados, on=['año', 'mes', 'region', 'comuna'], how='left')
df_master.rename(columns={'comuna': 'region_comuna'}, inplace=True)

# 2. Agregar el flag espacial exigido por la arquitectura de la red neuronal global
# IS_COMUNA viene definido como 0 desde la Celda 1
df_master['is_comuna'] = IS_COMUNA

# 3. Control de Calidad: Verificar si hubo pérdidas en el cruce de datos
filas_nulas = df_master['share_R'].isna().sum()
if filas_nulas > 0:
    print(f"⚠️ ¡Advertencia! Hay {filas_nulas} registros horarias que no encontraron su share anual.")
    print("Esto suele pasar si los nombres de las regiones difieren entre las bases (ej. tildes o alias).")
    # Opcional: llenar con 0 o con el promedio si deseas forzar la ejecución
    # df_master = df_master.fillna(0) 
else:
    print("✅ Cruce de datos completado exitosamente. Cero registros nulos encontrados.")

# 4. Validación estructural del orden del Vector de Entrada (Matriz X)
print("\nValidando alineación con la lista exigida por la Red Neuronal (columns.txt)...")

# Buscamos si hay alguna columna en columns.txt que no exista en nuestro DataFrame fusionado
columnas_faltantes = [col for col in columns if col not in df_master.columns]

if columnas_faltantes:
    print(f"❌ ERROR CRÍTICO: El modelo exige columnas que no están en el DataFrame: {columnas_faltantes}")
    print("👉 Consejo: Revisa si hay discrepancias de nombres (ej. 'hour_sin' vs 'sin_hora' o 'temp_t - 1' vs 'temp_t-1').")
    print("Modifica el nombre en el DataFrame para que calce exactamente con lo que pide columns.txt")
else:
    print(f"✅ Alineación perfecta. Las {len(columns)} variables requeridas están presentes.")
    
    # 5. Extracción de la Matriz X Definitiva (Tensor Ciego para Keras)
    # Al pasar la lista 'columns', forzamos a que las columnas queden en el ORDEN EXACTO en que se entrenó la red
    X_inferencia = df_master[columns].values
    
    print(f"\n⚡ ¡Matriz X construida con éxito!")
    print(f"-> Dimensiones finales del input para model.predict(): {X_inferencia.shape}")


Iniciando fusión global de shares proyectados y características temporales...
✅ Cruce de datos completado exitosamente. Cero registros nulos encontrados.

Validando alineación con la lista exigida por la Red Neuronal (columns.txt)...
✅ Alineación perfecta. Las 24 variables requeridas están presentes.

⚡ ¡Matriz X construida con éxito!
-> Dimensiones finales del input para model.predict(): (6052680, 24)


### Agregar parámetros de estandariación y desagregación

In [10]:
def _get_hours_in_year(year):
    """
    Determina la cantidad exacta de horas en un año (bisiesto o normal).
    """
    # Se emplea el método .isleap(year) que entrega un booleano si es bisiesto o no
    is_leap = calendar.isleap(year)
    return 8784 if is_leap else 8760


def calculate_scaling_parameters(total_anual_mwh, consumos_anuales_sectores, year, a, b):
    """
    Calcula mu y sigma global de la zona, así como los valores 'sin sector' 
    para la etapa de desagregación de la demanda.
    
    Args:
        total_anual_mwh (float): Consumo físico total proyectado para el año completo (MWh).
        consumos_anuales_sectores (dict): Diccionario con el consumo anual de cada sector.
                                          Ej: {'I': 1500.5, 'R': 2000.0, 'C': 500.0, ...}
        year (int): Año a evaluar (para definir 8760 vs 8784 horas).
        a (float): Parámetro 'a' de la curva de correlación empírica (sigma = a * mu^b).
        b (float): Parámetro 'b' de la curva de correlación empírica.
        
    Returns:
        dict: Diccionario que contiene mu_total, sigma_total, mu_sin_X, sigma_sin_X.
    """
    horas_anio = _get_hours_in_year(year)
    
    # 1. Parámetros Globales (Total de la Zona)
    mu_total = total_anual_mwh / horas_anio
    sigma_total = a * (mu_total ** b)
    
    resultados = {
        'mu_total': mu_total,
        'sigma_total': sigma_total
    }
    
    # 2. Parámetros "Sin Sector" para los Escenarios Puros (Desagregación)
    for sector, consumo_sector in consumos_anuales_sectores.items():
        # Restamos el consumo del sector al total. Aplicamos max(0.0) como filtro 
        # físico por si la matemática de proyección arrojara inconsistencias ínfimas.
        total_sin_sector = max(0.0, total_anual_mwh - consumo_sector)
        
        mu_sin = total_sin_sector / horas_anio
        
        # Omitimos cálculo de sigma si mu es 0 (ej. zona sin consumo) para evitar errores matemáticos
        if mu_sin > 0:
            sigma_sin = a * (mu_sin ** b)
        else:
            sigma_sin = 0.0
            
        resultados[f'mu_sin_{sector}'] = mu_sin
        resultados[f'sigma_sin_{sector}'] = sigma_sin
        
    return resultados

In [15]:
# ==========================================
# CELDA 5: CÁLCULO DE MU Y SIGMA (MACRO COMUNAL) Y MERGE A DF_MASTER
# ==========================================

print("Calculando parámetros de escalamiento (mu, sigma) por comuna y año...")

# 1. Reconstruir los consumos en MWh por sector y mes
# Hacemos una copia para no alterar el dataframe original en este paso
df_temp = df_shares_proyectados.copy()

for s in SECTORES:
    # Consumo sectorial mensual = Total mensual * Share del mes
    df_temp[f'consumo_{s}_MWh'] = df_temp['consumo_total_mes_MWh'] * df_temp[f'share_{s}']

# 2. Agrupar por Comuna y Año para obtener la matriz MACRO ANUAL
# Sumamos los 12 meses para obtener el consumo físico real de todo el año
cols_a_sumar = ['consumo_total_mes_MWh'] + [f'consumo_{s}_MWh' for s in SECTORES]

df_macro = df_temp.groupby(['año', 'comuna'])[cols_a_sumar].sum().reset_index()

# Renombramos la columna para que tenga coherencia semántica
df_macro.rename(columns={'consumo_total_mes_MWh': 'total_consumo_anual_MWh'}, inplace=True)

# 3. Función envoltorio para aplicar el cálculo fila por fila
def apply_scaling(row):
    year = int(row['año'])
    
    # El total ya está sumado anualmente y en MWh (ya no multiplicamos por 1000)
    total_mwh = row['total_consumo_anual_MWh']
    
    # Capturar los consumos anuales por sector ya sumados
    consumos_sectores = {s: row[f'consumo_{s}_MWh'] for s in SECTORES}
    
    # Llamamos a tu función intacta
    return calculate_scaling_parameters(
        total_anual_mwh=total_mwh,
        consumos_anuales_sectores=consumos_sectores,
        year=year,
        a=A_PARAM, # Asegúrate de que estén definidos en tu celda de importaciones
        b=B_PARAM
    )

# 4. Aplicar y expandir los diccionarios resultantes en nuevas columnas
df_params = df_macro.apply(apply_scaling, axis=1, result_type='expand')

# Concatenamos los parámetros calculados a nuestra tabla macro
# NOTA: Cambiamos 'region_bne' por 'region_comuna'
df_macro_completa = pd.concat([df_macro[['año', 'comuna']], df_params], axis=1)
df_macro_completa.rename(columns={'comuna': 'region_comuna'}, inplace=True)

# 5. Unir (Merge) estos parámetros de vuelta a la base horaria (df_master)
df_master = pd.merge(df_master, df_macro_completa, on=['año', 'region_comuna'], how='left')

print(f"✅ ¡Parámetros comunales agregados con éxito! Dimensiones de df_master: {df_master.shape}")

Calculando parámetros de escalamiento (mu, sigma) por comuna y año...
✅ ¡Parámetros comunales agregados con éxito! Dimensiones de df_master: (6052680, 50)


### Inferencia

In [16]:
def predict_and_disaggregate(model, df_inputs, df_metadata, feature_cols, sectores=['I', 'R', 'C', 'P', 'T']):
    """
    Realiza la estimación de demanda eléctrica total y la desagregación sectorial 
    mediante el método de resta de escenarios.
    
    Args:
        model (keras.Model): Modelo MLP global pre-entrenado.
        df_inputs (pd.DataFrame): DataFrame solo con los features (X) numéricos y escalados.
        df_metadata (pd.DataFrame): DataFrame con metadatos asociados a las filas de df_inputs 
                                    (debe contener mu_total, sigma_total, mu_sin_X, sigma_sin_X).
        feature_cols (list): Lista con el orden exacto de las columnas que traga el modelo.
        sectores (list): Lista de los identificadores de los sectores.
        
    Returns:
        pd.DataFrame: df_metadata enriquecido con las curvas de demanda total y por sector en MWh.
    """
    df_res = df_metadata.copy()
    
    # Aseguramos el orden estricto de las columnas para la red neuronal
    X_total = df_inputs[feature_cols].values

    # ---------------------------------------------------------
    # PASO 1: PREDICCIÓN TOTAL
    # ---------------------------------------------------------
    print("-> Calculando demanda total...")
    y_pred_scaled = model.predict(X_total, batch_size=2048).flatten()
    
    # Desescalamiento con parámetros globales
    df_res['demanda_total_pred'] = (y_pred_scaled * df_res['sigma_total']) + df_res['mu_total']
    df_res['demanda_total_pred'] = df_res['demanda_total_pred'].clip(lower=0.0) # Filtro físico

    # ---------------------------------------------------------
    # PASO 2: DESAGREGACIÓN SECTORIAL (MÉTODO DE RESTA)
    # ---------------------------------------------------------
    for sector in sectores:
        print(f"-> Desagregando sector: {sector}")
        df_sim = df_inputs.copy()
        
        # 1. Apagar el sector objetivo (llevar su share a cero)
        col_share = f'share_{sector}'
        df_sim[col_share] = 0.0

        # 2. Predecir el escenario "Sin el Sector"
        X_sim = df_sim[feature_cols].values
        y_pred_sin_scaled = model.predict(X_sim, batch_size=2048).flatten()

        # 3. Desescalar utilizando mu y sigma SIN el sector
        col_mu_sin = f'mu_sin_{sector}'
        col_sigma_sin = f'sigma_sin_{sector}'
        
        demanda_sin_pred = (y_pred_sin_scaled * df_res[col_sigma_sin]) + df_res[col_mu_sin]
        demanda_sin_pred = demanda_sin_pred.clip(lower=0.0)

        # 4. Obtener demanda del sector por diferencia (Total - Predicción sin el sector)
        col_demanda_sector = f'demanda_pred_{sector}'
        df_res[col_demanda_sector] = df_res['demanda_total_pred'] - demanda_sin_pred
        df_res[col_demanda_sector] = df_res[col_demanda_sector].clip(lower=0.0)

    return df_res

In [19]:
# ==========================================
# CELDA 6: INFERENCIA Y MÉTODO KUSUMOTO (ESCENARIOS PUROS)
# ==========================================
print("Iniciando motor de desagregación sectorial (Red Neuronal)...")

# 1. Separar los Inputs estrictos (Matriz X Ciega)
# La lista 'columns' viene del txt cargado en la Celda 1. Esto garantiza orden perfecto.
df_inputs = df_master[columns].copy()

# 2. Separar la Metadata (Lo que usaremos para desescalar y trazar resultados)
cols_metadata = ['fecha_hora', 'año', 'region_comuna', 'region', 'comuna_gdf', 'region_gdf'] + \
                ['mu_total', 'sigma_total'] + \
                [f'mu_sin_{s}' for s in SECTORES] + \
                [f'sigma_sin_{s}' for s in SECTORES]

df_metadata = df_master[cols_metadata].copy()

# 3. Ejecutar la función de inferencia (modelo cargado previamente)
df_resultados = predict_and_disaggregate(
    model=model, 
    df_inputs=df_inputs, 
    df_metadata=df_metadata, 
    feature_cols=columns,     # Le pasamos la lista para que X_sim no se desordene 
    sectores=SECTORES
)

print("\n⚡ ¡Inferencia y Desagregación Completadas!")
print("Muestra de los resultados (Total y Sectores en MWh):")
print(df_resultados[['fecha_hora', 'region_comuna', 'demanda_total_pred'] + [f'demanda_pred_{s}' for s in SECTORES]].head())

Iniciando motor de desagregación sectorial (Red Neuronal)...
-> Calculando demanda total...
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
-> Desagregando sector: R
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
-> Desagregando sector: C
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
-> Desagregando sector: P
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
-> Desagregando sector: I
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
-> Desagregando sector: T
2956/2956 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step

⚡ ¡Inferencia y Desagregación Completadas!
Muestra de los resultados (Total y Sectores en MWh):
           fecha_hora region_comuna  demanda_total_pred  demanda_pred_R  \
0 2024-01-01 00:00:00       IQUIQUE           35.259785       19.631230   
1 2024-01-01 01:00:00       IQUIQUE           31.972399       17.121471   
2 2024-01-01 02:00:00       IQUIQUE           29.253929       15.044523   
3 2024-01-01 03:00:00       IQUIQUE           27.234811       13.608340   
4 2024-01-01 04:00:00       IQUIQUE           25

In [22]:
import pandas as pd
import geopandas as gpd
import os

# --- 0. CONFIGURACIÓN INICIAL ---
SECTORES = ['R', 'C', 'P', 'I', 'T']
# Rutas de salida (ajusta según tu estructura real)
OUTPUT_PARQUET = os.path.join(OUT_DIR, "wp2_output_demanda_electrica_comunal_ts.parquet")
OUTPUT_GPKG = os.path.join(OUT_DIR, "wp2_output_demanda_electrica_comunal.gpkg")
RUTA_CAPA_BASE = LAYER_PATH  # Tu mapa base con las geometrías de las regiones

# --- 1. AJUSTES AL DATAFRAME DE RESULTADOS ---
# Renombrar de vuelta 'region_comuna' a 'region' y asegurar el nombre de la fecha a 'timestamp'
df_resultados.drop(columns='region', inplace=True)
df_resultados = df_resultados.rename(columns={
    'region_gdf': 'region',
    'comuna_gdf': 'comuna',
    'fecha_hora': 'timestamp' # Si ya se llamaba timestamp, esto no afecta
})

# Asegurarse de que el timestamp sea un objeto datetime de pandas
df_resultados['timestamp'] = pd.to_datetime(df_resultados['timestamp'])

# --- 2. SERIE DE TIEMPO (.parquet) ---
print("Generando serie de tiempo horaria...")

# Definir las columnas exactas que solicitaste para la serie de tiempo
cols_sectores = [f"demanda_pred_{sector}" for sector in SECTORES]
cols_ts = ["timestamp", "region", "comuna", "demanda_total_pred"] + cols_sectores

# Filtrar el DataFrame
df_time_series = df_resultados[cols_ts].copy()
cols_sectores_mwh = [f"demanda_{sector}_MWh" for sector in SECTORES]
rename_cols = {f"demanda_pred_{sector}": f"demanda_{sector}_mwh" for sector in SECTORES}
rename_cols["demanda_total_pred"] = "demanda_MWh"
df_time_series = df_time_series.rename(columns=rename_cols)

# Guardar en parquet para el equipo de ingesta
os.makedirs(os.path.dirname(OUTPUT_PARQUET), exist_ok=True)
df_time_series.to_parquet(OUTPUT_PARQUET, engine='pyarrow', index=False)
print(f"-> Serie de tiempo guardada en: {OUTPUT_PARQUET}")


# --- 3. AGREGACIÓN ANUAL Y GEOCAPA (.gpkg) ---
print("\nCalculando totales anuales y generando geocapa...")

# Extraer el año para la agrupación
df_resultados['año'] = df_resultados['timestamp'].dt.year

# Agrupar por año y comuna, sumando los MWh de todo el año
df_anual_mwh = df_resultados.groupby(['año', 'region', 'comuna'])[["demanda_total_pred"] + cols_sectores].sum().reset_index()

# Conversión de unidades: MWh a GWh (1 GWh = 1,000 MWh)
df_anual_mwh['demanda_total_GWh'] = df_anual_mwh['demanda_total_pred'] / 1000.0

cols_sectores_gwh = []
for sector in SECTORES:
    col_mwh = f"demanda_pred_{sector}"
    col_gwh = f"demanda_{sector}_GWh"
    df_anual_mwh[col_gwh] = df_anual_mwh[col_mwh] / 1000.0
    cols_sectores_gwh.append(col_gwh)

# Seleccionar solo las columnas objetivo para la geocapa
cols_geocapa = ["año", "region", "comuna", "demanda_total_GWh"] + cols_sectores_gwh
df_datos_geocapa = df_anual_mwh[cols_geocapa].copy()

# Cargar la capa vectorial (GeoDataFrame) de regiones
# IMPORTANTE: Asegúrate de que esta capa tenga una columna llamada 'region' con los mismos nombres
gdf_regiones = gpd.read_file(RUTA_CAPA_BASE)

# Unir los datos anuales con las geometrías (Merge)
gdf_final = gdf_regiones.merge(df_datos_geocapa, on="comuna", how="inner")

# Guardar como GeoPackage
os.makedirs(os.path.dirname(OUTPUT_GPKG), exist_ok=True)
gdf_final.to_file(OUTPUT_GPKG, driver="GPKG")
print(f"-> Geocapa anual guardada exitosamente en: {OUTPUT_GPKG}")

Generando serie de tiempo horaria...
-> Serie de tiempo guardada en: /home/ica/MERLIN_EDM/prototipo_3/data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal_ts.parquet

Calculando totales anuales y generando geocapa...
-> Geocapa anual guardada exitosamente en: /home/ica/MERLIN_EDM/prototipo_3/data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal.gpkg
